# Table 3 and Supplementary Table S2 — parameters for both fits and all four cases

Python version of `Table3_S2_parameters_both_calibrations.m`.

* **Including the first day** (main fit): days 0–4, every series normalized to its day‑0 value.
* **Excluding the first day**: days 1–4, every series normalized to its day‑1 value, all parameters re‑estimated.

For each fit: monoculture $\alpha$ and $\delta$ (replicate bootstrap), then the four co‑culture cases $H_0$, $H_\beta$, $H_\gamma$, $H_{\beta\gamma}$ with AICc, residual‑bootstrap 95% CIs, the interior point $x^*=(\beta-\delta)/[(\beta-\delta)+(\gamma-\alpha)]$, the game class, the share of bootstrap fits in the same class, and the stable outcome for the observed initial leader fractions.

**Table 3** (main text) = the best case of each fit; **Table S2** = the remaining cases.

The shared routines are in `calibration.py`. Bootstrap CIs agree with the MATLAB output up to resampling noise (second or third decimal). Runtime: about 40 minutes with `N_BOOT = 1000` on 2 cores (eight bootstraps, run in parallel).

In [1]:
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
import calibration as cal

N_BOOT = 1000                 # bootstrap samples per case (100 for a quick test)
N_WORKERS = 4                 # parallel processes for the bootstrap (1 = no parallelism)
FITS = {0: "Including the first day (days 0-4)", 1: "Excluding the first day (days 1-4)"}

## Monocultures and best fits of every case

In [2]:
R = {}
for start in (0, 1):
    rng = np.random.default_rng(1 + start)
    alpha, delta, M = cal.fit_monocultures(start, N_BOOT, rng)
    S, W = cal.coculture_data(start, alpha, delta)
    n = 2 * (len(S[0]["x"]) - 1) * len(S)
    fits = {c: cal.fit_case(c, S, W) for c in cal.CASES}
    aic = {c: cal.aicc(fits[c][1], n, cal.CASE_K[c]) for c in cal.CASES}
    R[start] = dict(alpha=alpha, delta=delta, M=M, S=S, W=W, n=n, fits=fits,
                    dAIC={c: aic[c] - min(aic.values()) for c in cal.CASES},
                    best=min(aic, key=aic.get), x0max=max(s["x"][0] for s in S))
    print(f"\n{FITS[start]}: n = {n} fitted points, largest initial leader fraction x0 = {R[start]['x0max']:.3f}")
    for (ct, c), m in M.items():
        print(f"  {ct:9s} {cal.COND[c]:11s} {m['r_mean']:.4f} [{m['r_ci'][0]:.4f}, {m['r_ci'][1]:.4f}]")
    print("  dAICc:", {c: round(float(v), 2) for c, v in R[start]["dAIC"].items()}, "-> best case", R[start]["best"])


Including the first day (days 0-4): n = 48 fitted points, largest initial leader fraction x0 = 0.517
  Leaders   untreated   0.3788 [0.3340, 0.4186]
  Leaders   +TGF-beta1  0.1202 [0.0568, 0.1653]
  Followers untreated   0.4476 [0.4145, 0.4683]
  Followers +TGF-beta1  0.4440 [0.3970, 0.4800]
  dAICc: {'H0': 0.0, 'Hbeta': 2.3, 'Hgamma': 2.34, 'Hbg': 4.75} -> best case H0



Excluding the first day (days 1-4): n = 36 fitted points, largest initial leader fraction x0 = 0.447
  Leaders   untreated   0.3658 [0.3030, 0.4119]
  Leaders   +TGF-beta1  0.0830 [0.0278, 0.1188]
  Followers untreated   0.5730 [0.4994, 0.6227]
  Followers +TGF-beta1  0.5715 [0.4427, 0.6838]
  dAICc: {'H0': 3.76, 'Hbeta': 5.96, 'Hgamma': 0.0, 'Hbg': 1.64} -> best case Hgamma


## Residual bootstrap of every case (both fits, in parallel)

In [3]:
jobs = [(start, c) for start in (0, 1) for c in cal.CASES]
args = [(c, R[s]["fits"][c][0], R[s]["S"], R[s]["W"], N_BOOT, 100 + 10 * s + cal.CASES.index(c)) for s, c in jobs]
if N_WORKERS > 1:
    with ProcessPoolExecutor(N_WORKERS) as pool:
        draws = list(pool.map(cal.bootstrap_job, args))
else:
    draws = [cal.bootstrap_job(a) for a in args]
BOOT = dict(zip(jobs, draws))
print("bootstrap fits per case:", {f"{s}-{c}": len(P) for (s, c), P in BOOT.items()})

bootstrap fits per case: {'0-H0': 1000, '0-Hbeta': 1000, '0-Hgamma': 1000, '0-Hbg': 1000, '1-H0': 1000, '1-Hbeta': 1000, '1-Hgamma': 1000, '1-Hbg': 1000}


## Results: parameters, $x^*$, game and outcome

In [4]:
rows = []
for start in (0, 1):
    r = R[start]
    for c in cal.CASES:
        q, P = r["fits"][c][0], BOOT[(start, c)]
        lo, hi = np.percentile(P, [2.5, 97.5], 0)
        for k, cond in enumerate((1, 2)):
            a, d = r["alpha"][k], r["delta"][k]
            b, g = q[2 * k], q[2 * k + 1]
            xs, gm = cal.xstar(a, b, g, d), cal.game(a, b, g, d)
            xb = np.array([cal.xstar(a, p[2 * k], p[2 * k + 1], d) for p in P])
            same = 100 * np.mean([cal.game(a, p[2 * k], p[2 * k + 1], d) == gm for p in P])
            interior = gm in ("Coordination", "Snowdrift")
            rows.append({
                "fit": FITS[start], "case": c + ("  (best)" if c == r["best"] else ""),
                "dAICc": round(r["dAIC"][c], 2), "condition": cal.COND[cond],
                "beta [95% CI]": f"{b:.3f} [{lo[2*k]:.3f}, {hi[2*k]:.3f}]",
                "gamma [95% CI]": f"{g:.3f} [{lo[2*k+1]:.3f}, {hi[2*k+1]:.3f}]",
                "x*": (f"{xs:.2f} [{np.percentile(xb, 2.5):.2f}, {np.percentile(xb, 97.5):.2f}]"
                       if interior else f"{xs:.1f} (outside [0,1])"),
                "game": gm, "same class (%)": round(same, 1),
                "outcome": cal.outcome(gm, xs, r["x0max"])})
table = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None, "display.width", 250)
table

,fit,case,dAICc,condition,beta [95% CI],gamma [95% CI],x*,game,same class (%),outcome
0,Including the first day (days 0-4),H0 (best),0.00,untreated,"0.124 [0.085, 0.153]","0.764 [0.685, 0.826]","-5.3 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
1,Including the first day (days 0-4),H0 (best),0.00,+TGF-beta1,"0.124 [0.085, 0.153]","0.764 [0.685, 0.826]","-1.0 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
2,Including the first day (days 0-4),Hbeta,2.30,untreated,"0.130 [0.077, 0.170]","0.764 [0.691, 0.823]","-4.7 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
3,Including the first day (days 0-4),Hbeta,2.30,+TGF-beta1,"0.114 [0.063, 0.158]","0.764 [0.691, 0.823]","-1.1 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
4,Including the first day (days 0-4),Hgamma,2.34,untreated,"0.124 [0.084, 0.154]","0.757 [0.663, 0.836]","-6.0 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
5,Including the first day (days 0-4),Hgamma,2.34,+TGF-beta1,"0.124 [0.084, 0.154]","0.773 [0.645, 0.869]","-1.0 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
6,Including the first day (days 0-4),Hbg,4.75,untreated,"0.130 [0.080, 0.171]","0.757 [0.658, 0.833]","-5.2 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
7,Including the first day (days 0-4),Hbg,4.75,+TGF-beta1,"0.114 [0.055, 0.156]","0.774 [0.640, 0.865]","-1.0 (outside [0,1])",Prisoner's Dilemma,100.0,"x -> 0 (followers only, from any x0)"
8,Excluding the first day (days 1-4),H0,3.76,untreated,"-0.151 [-0.196, -0.111]","-0.097 [-0.166, -0.036]","0.61 [0.59, 0.64]",Coordination,100.0,x -> 0 (followers only: all x0 <= 0.45 < x* = 0.61)
9,Excluding the first day (days 1-4),H0,3.76,+TGF-beta1,"-0.151 [-0.196, -0.111]","-0.097 [-0.166, -0.036]","0.80 [0.75, 0.85]",Coordination,100.0,x -> 0 (followers only: all x0 <= 0.45 < x* = 0.80)


In [5]:
print("TABLE 3 (main text): monocultures + best case of each fit")
for start in (0, 1):
    r = R[start]; M = r["M"]
    print(f"\n{FITS[start]}")
    for (ct, c), lab in ((("Leaders", 1), "alpha untreated"), (("Leaders", 2), "alpha +TGF"),
                         (("Followers", 1), "delta (shared)"), (("Followers", 2), "delta +TGF (not used)")):
        print(f"  {lab:22s} {M[(ct, c)]['r_mean']:.4f} [{M[(ct, c)]['r_ci'][0]:.4f}, {M[(ct, c)]['r_ci'][1]:.4f}]")
    print(table[(table.fit == FITS[start]) & table.case.str.contains("best")].drop(columns="fit").to_string(index=False))
print("\nTABLE S2 (supplementary): remaining cases")
print(table[~table.case.str.contains("best")].to_string(index=False))

TABLE 3 (main text): monocultures + best case of each fit

Including the first day (days 0-4)
  alpha untreated        0.3788 [0.3340, 0.4186]
  alpha +TGF             0.1202 [0.0568, 0.1653]
  delta (shared)         0.4476 [0.4145, 0.4683]
  delta +TGF (not used)  0.4440 [0.3970, 0.4800]
      case  dAICc  condition        beta [95% CI]       gamma [95% CI]                   x*               game  same class (%)                              outcome
H0  (best)    0.0  untreated 0.124 [0.085, 0.153] 0.764 [0.685, 0.826] -5.3 (outside [0,1]) Prisoner's Dilemma           100.0 x -> 0 (followers only, from any x0)
H0  (best)    0.0 +TGF-beta1 0.124 [0.085, 0.153] 0.764 [0.685, 0.826] -1.0 (outside [0,1]) Prisoner's Dilemma           100.0 x -> 0 (followers only, from any x0)

Excluding the first day (days 1-4)
  alpha untreated        0.3658 [0.3030, 0.4119]
  alpha +TGF             0.0830 [0.0278, 0.1188]
  delta (shared)         0.5730 [0.4994, 0.6227]
  delta +TGF (not used)  0.5715 [0.